In [ ]:
import pandas as pd
import pickle
import arviz as az
from plotly import graph_objects as go
from datetime import datetime
import pycountry
import numpy as np

from emu_renewal.inputs import ANALYSIS_TYPES, DATA_PATH, BASE_PATH
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_spaghetti_calib_comparison, plot_post_prior_comparison, plot_imm_props, get_google_mobility

In [ ]:
country = "France"
analysis = "no_mob"
run_name = "43485145"

In [ ]:
path = BASE_PATH / "outputs" / run_name / pycountry.countries.lookup(country).alpha_3 / analysis
spaghetti = pd.read_hdf(path / "spaghetti.h5")
targets = load_targets(path)
idata = az.from_netcdf(path / "idata_filtered.nc")

In [ ]:
outputs = list(targets.keys()) + ["process", "seropos"]
spagh_plot = plot_spaghetti_calib_comparison(spaghetti, targets, outputs)
mobility = get_google_mobility(pycountry.countries.lookup(country).alpha_3)
filtered_mob = mobility.loc[mobility.index < spaghetti.index[-1]]
spagh_plot.add_traces(go.Scatter(x=filtered_mob.index, y=filtered_mob["retail_and_recreation"], line={"color": "blue"}), rows=outputs.index("process") + 1, cols=1)

In [ ]:
plot_imm_props(spaghetti)

In [ ]:
priors = pickle.load(open(path / "priors.pkl", "rb"))
epi_params = [p for p in priors if p != "shared_dispersion"]
n_rows = int(np.ceil(len(priors) / 2))
plot_post_prior_comparison(idata, epi_params, priors, req_grid=[n_rows, 2], req_size=[10, 40]);